# **Collaborative Filtering**

Collaborative filtering is a way to predict possible choices of people based on their prev actions and the actions of people who share similar taste of choices on that particular subject. Say the ratings given by two diff people on a set of movies are closely similar, then given that one person likes a particular movie then most likely the other person would like it too because of their shared similarity in ratingsa nd choices.

# Memory Based(Neighbourhood methods)

**User-based**: Lets say you wanna predict Alice's rating for a given movie.Then you'll find a set of similar people who share choices with Alice based on prev choices and find the avg of their choices for this given movie.Now Alice will be predicted to have given this calculated rating to the movie.

**Item-based:** Find the movies that are rated similar to this given movie across all viewers and find what rating was given by Alice for those movies and take average, then this movie will also be most likely rated very similarly to the avg.

In [3]:
import os
import urllib.request
import zipfile
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error

In [4]:
def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

For the dataset using "MovieLens-100k dataset" feels right. This dataset basically contains 100,000 ratings from 943 users across 1,682 movies and the ratings being on a scale of 1 to 5.The feed-back matrix has roughly 6.3% density only which is just as almost same as the real world case. This sparsity also helps fitting the data for processing and calculations.Also every user has rated atleats rated 20 movies ensuring memory-based neighbourhood dont fail cuz of empty histories.

In [5]:
def load_movielens_100k():
    data_dir = "./ml-100k"
    data_file = os.path.join(data_dir, "u.data")
    if not os.path.exists(data_file):
        print("downloading")
        url = "https://files.grouplens.org/datasets/movielens/ml-100k.zip"
        zip_path = "ml-100k.zip"
        urllib.request.urlretrieve(url, zip_path)
        with zipfile.ZipFile(zip_path, "r") as zip_ref:
            zip_ref.extractall(".")
        os.remove(zip_path)
    
    cols = ["user_id", "item_id", "rating", "timestamp"]
    df = pd.read_csv(data_file, sep="\t", names=cols, engine="python")
    
    df["user_idx"] = df["user_id"] - 1
    df["item_idx"] = df["item_id"] - 1
    
    return df

df = load_movielens_100k()
num_users = df["user_idx"].nunique()
num_items = df["item_idx"].nunique()
global_mean = df["rating"].mean()

print(f"Dataset: {len(df)} ratings | {num_users} users | {num_items} items")
print(f"Global Average Rating: {global_mean:.4f}")

train_df, test_df = train_test_split(df, test_size=0.20, random_state=42)
print(f"Train split: {len(train_df)} rows | Test split: {len(test_df)} rows")

Dataset: 100000 ratings | 943 users | 1682 items
Global Average Rating: 3.5299
Train split: 80000 rows | Test split: 20000 rows


In [6]:
class MemoryBasedItemCF:
    def __init__(self, k_neighbors=20):
        self.k = k_neighbors
        self.similarity_matrix = None
        self.user_item_matrix = None
        self.user_mean_ratings = None

    def fit(self, train_data, num_u, num_i):
        self.user_item_matrix = np.full((num_u, num_i), np.nan)
        for _, row in train_data.iterrows():
            self.user_item_matrix[int(row["user_idx"]), int(row["item_idx"])] = row["rating"]
        
        self.user_mean_ratings = np.nanmean(self.user_item_matrix, axis=1)
        self.user_mean_ratings = np.nan_to_num(self.user_mean_ratings, nan=global_mean)
        
        centered_matrix = self.user_item_matrix - self.user_mean_ratings[:, np.newaxis]
        centered_matrix = np.nan_to_num(centered_matrix, nan=0.0)
        
        norm = np.linalg.norm(centered_matrix, axis=0, keepdims=True)
        norm[norm == 0] = 1e-9
        normalized_items = centered_matrix / norm
        
        self.similarity_matrix = np.dot(normalized_items.T, normalized_items)
        np.fill_diagonal(self.similarity_matrix, 0.0)

    def predict_one(self, u_idx, i_idx):
        user_ratings = self.user_item_matrix[u_idx]
        rated_item_indices = np.where(~np.isnan(user_ratings))[0]
        
        if len(rated_item_indices) == 0:
            return self.user_mean_ratings[u_idx]
        
        sims = self.similarity_matrix[i_idx, rated_item_indices]
        
        positive_mask = sims > 0
        if not np.any(positive_mask):
            return self.user_mean_ratings[u_idx]
        
        top_k_indices = np.argsort(sims[positive_mask])[-self.k:]
        selected_sims = sims[positive_mask][top_k_indices]
        selected_ratings = user_ratings[rated_item_indices[positive_mask][top_k_indices]]
        
        denom = np.sum(np.abs(selected_sims))
        if denom == 0:
            return self.user_mean_ratings[u_idx]
        
        pred = np.sum(selected_sims * selected_ratings) / denom
        return float(np.clip(pred, 1.0, 5.0))

    def evaluate(self, test_data):
        preds = []
        actuals = test_data["rating"].values
        for _, row in test_data.iterrows():
            preds.append(self.predict_one(int(row["user_idx"]), int(row["item_idx"])))
        rmse = np.sqrt(mean_squared_error(actuals, preds))
        mae = mean_absolute_error(actuals, preds)
        return rmse, mae, preds

# **Model Based Matrix Factorisation**


In [7]:
class MovielensDataset(Dataset):
    def __init__(self, df):
        self.users = torch.tensor(df["user_idx"].values, dtype=torch.long)
        self.items = torch.tensor(df["item_idx"].values, dtype=torch.long)
        self.ratings = torch.tensor(df["rating"].values, dtype=torch.float32)

    def __len__(self):
        return len(self.ratings)

    def __getitem__(self, idx):
        return self.users[idx], self.items[idx], self.ratings[idx]

class MatrixFactorization(nn.Module):
    def __init__(self, num_users, num_items, global_mean, embedding_dim=32):
        super().__init__()
        self.global_mean = nn.Parameter(torch.tensor(global_mean, dtype=torch.float32), requires_grad=False)
        self.user_embedding = nn.Embedding(num_users, embedding_dim)
        self.item_embedding = nn.Embedding(num_items, embedding_dim)
        self.user_bias = nn.Embedding(num_users, 1)
        self.item_bias = nn.Embedding(num_items, 1)
        
        nn.init.normal_(self.user_embedding.weight, std=0.01)
        nn.init.normal_(self.item_embedding.weight, std=0.01)
        nn.init.zeros_(self.user_bias.weight)
        nn.init.zeros_(self.item_bias.weight)

    def forward(self, u, i):
        u_lat = self.user_embedding(u) 
        i_lat = self.item_embedding(i)
        dot_product = (u_lat * i_lat).sum(dim=1)
        
        b_u = self.user_bias(u).squeeze(1)
        b_i = self.item_bias(i).squeeze(1)
        
        pred = self.global_mean + b_u + b_i + dot_product
        return pred

def train_matrix_factorization(model, train_loader, val_df, epochs=15, lr=0.005, weight_decay=1e-4):
    device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
    model = model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.MSELoss()
    
    val_u = torch.tensor(val_df["user_idx"].values, dtype=torch.long).to(device)
    val_i = torch.tensor(val_df["item_idx"].values, dtype=torch.long).to(device)
    val_r = val_df["rating"].values

    history = {"train_loss": [], "val_rmse": []}

    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        for u, i, r in train_loader:
            u, i, r = u.to(device), i.to(device), r.to(device)
            optimizer.zero_grad()
            preds = model(u, i)
            loss = criterion(preds, r)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(r)

        train_loss = total_loss / len(train_loader.dataset)
        
        model.eval()
        with torch.no_grad():
            val_preds = model(val_u, val_i).cpu().numpy()
            val_preds = np.clip(val_preds, 1.0, 5.0)
            val_rmse = np.sqrt(mean_squared_error(val_r, val_preds))
            val_mae = mean_absolute_error(val_r, val_preds)

        history["train_loss"].append(train_loss)
        history["val_rmse"].append(val_rmse)
        print(f"Epoch {epoch+1:02d}/{epochs:02d} | Train MSE: {train_loss:.4f} | Val RMSE: {val_rmse:.4f} | Val MAE: {val_mae:.4f}")

    return history, val_preds

# Execution and Benchmarking

In [8]:
print("\n--- Training Memory-Based Item-CF ---")
item_cf = MemoryBasedItemCF(k_neighbors=25)
item_cf.fit(train_df, num_users, num_items)
mem_rmse, mem_mae, mem_preds = item_cf.evaluate(test_df)
print(f"Memory-Based Item-CF -> Test RMSE: {mem_rmse:.4f} | Test MAE: {mem_mae:.4f}")

print("\n--- Training Model-Based Matrix Factorization ---")
train_dataset = MovielensDataset(train_df)
train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
mf_model = MatrixFactorization(num_users, num_items, global_mean, embedding_dim=32)
history, mf_preds = train_matrix_factorization(mf_model, train_loader, test_df, epochs=15, lr=0.005, weight_decay=1e-3)
mf_rmse = np.sqrt(mean_squared_error(test_df["rating"].values, mf_preds))
mf_mae = mean_absolute_error(test_df["rating"].values, mf_preds)
print(f"Model-Based Matrix Factorization -> Test RMSE: {mf_rmse:.4f} | Test MAE: {mf_mae:.4f}")


--- Training Memory-Based Item-CF ---
Memory-Based Item-CF -> Test RMSE: 0.9266 | Test MAE: 0.7245

--- Training Model-Based Matrix Factorization ---
Epoch 01/15 | Train MSE: 1.0509 | Val RMSE: 0.9394 | Val MAE: 0.7425
Epoch 02/15 | Train MSE: 0.7360 | Val RMSE: 0.9229 | Val MAE: 0.7261
Epoch 03/15 | Train MSE: 0.5326 | Val RMSE: 0.9512 | Val MAE: 0.7457
Epoch 04/15 | Train MSE: 0.4085 | Val RMSE: 0.9848 | Val MAE: 0.7687
Epoch 05/15 | Train MSE: 0.3409 | Val RMSE: 1.0125 | Val MAE: 0.7902
Epoch 06/15 | Train MSE: 0.3008 | Val RMSE: 1.0376 | Val MAE: 0.8073
Epoch 07/15 | Train MSE: 0.2737 | Val RMSE: 1.0564 | Val MAE: 0.8210
Epoch 08/15 | Train MSE: 0.2553 | Val RMSE: 1.0728 | Val MAE: 0.8328
Epoch 09/15 | Train MSE: 0.2410 | Val RMSE: 1.0859 | Val MAE: 0.8419
Epoch 10/15 | Train MSE: 0.2298 | Val RMSE: 1.0990 | Val MAE: 0.8502
Epoch 11/15 | Train MSE: 0.2208 | Val RMSE: 1.1104 | Val MAE: 0.8587
Epoch 12/15 | Train MSE: 0.2141 | Val RMSE: 1.1209 | Val MAE: 0.8663
Epoch 13/15 | Train M